## **Load Libraries**

In [23]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tabulate import tabulate

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

# Download components
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## **Load Data**

In [24]:
sales = pd.read_csv("/content/drive/MyDrive/CIND 820/Milestone 4/sales_compressed.csv")

## **Treatment of Outliers**

In [25]:
# Custom transformer to impute outliers in classification pipeline
class OutlierReplacement(BaseEstimator, TransformerMixin):
    def __init__(self, reg_indicator=False):
        self.reg_indicator = reg_indicator # True for regression, otherwise false
        self.stats = {} # Dictionary of DataFrames to store data for each feature by category
        '''
        Example structure:
        self.stats = {'actual_price': {'min_val': [1, 2, 3],
                                       'max_val': [4, 5, 6],
                                       'mean': [7, 8, 9],
                                       'category': ['cat1', 'cat2', 'cat3']},
                      'discounted_price': {'min_val': [1, 2, 3],
                                              'max_val': [4, 5, 6],
                                              'mean': [7, 8, 9],
                                              'category': ['cat1', 'cat2', 'cat3']}, ...}
        '''
        # Determine which set of features to use based on the model
        if reg_indicator:
          self.numerical_features = ['actual_price', 'discount_percentage', 'rating_count', 'rating']
        else:
          self.numerical_features = ['actual_price', 'discounted_price', 'discount_percentage', 'rating_count', 'rating']

    # Perform necessary calculations using only the training data to prevent data leakage
    def fit(self, X, y):
       # Calculate the IQR and mean for every featrure by category
        for feature in self.numerical_features:
            category_stats = X.groupby("category")[feature].agg(
                q1=lambda x: x.quantile(0.25),
                q3=lambda x: x.quantile(0.75),
                iqr=(lambda x: x.quantile(0.75) - x.quantile(0.25)),
                mean='mean'
            )
            category_stats['min_val'] = category_stats['q1'] - 1.5 * category_stats['iqr']
            category_stats['max_val'] = category_stats['q3'] + 1.5 * category_stats['iqr']

            # Store necessary values to detect outliers
            self.stats[feature] = category_stats[['min_val', 'max_val', 'mean']]

        return self

    # Transform the training and test data
    def transform(self, X):
        categories = X["category"]

        # Impute outliers using category calculations
        for feature in self.numerical_features:
            feature_stats = self.stats[feature]

            # Map category stats to each row of the data
            cat_min_vals = categories.map(feature_stats['min_val'])
            cat_max_vals = categories.map(feature_stats['max_val'])
            cat_mean_vals = categories.map(feature_stats['mean'])

            # Create a boolean mask to detect outliers
            # Compare each row's feature value to the corresponding min/max for its category
            outliers = (X[feature] < cat_min_vals) | (X[feature] > cat_max_vals)

            # Replace outliers using category mean where boolean mask is true
            X[feature] = np.where(outliers, cat_mean_vals, X[feature])

        # Remove target from data before modeling phase
        return X[self.numerical_features]

## **Multinomial Logistic Regression**

In [26]:
# Build Pipeline to process data separately during each fold
pipeline = Pipeline([
    ('imputation', OutlierReplacement(reg_indicator=True)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

# Perform cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = cross_validate(pipeline, sales, sales['category'], cv=cv,
                            scoring=['accuracy', 'precision_weighted', 'recall_weighted'],
                            return_train_score=False)

# Extract scores
reg_train_time = results['fit_time'].mean()
reg_test_time = results['score_time'].mean()
reg_accuracy = results['test_accuracy'].mean()
reg_precision = results['test_precision_weighted'].mean()
reg_recall = results['test_recall_weighted'].mean()

# Output scores
table_rows = [
    ["Average Accuracy", reg_accuracy],
    ["Weighted Precision", reg_precision],
    ["Weighted Recall", reg_recall],
    ["Average Training Time (Seconds)", reg_train_time],
    ["Average Training Time (Seconds)", reg_test_time]
]

headers = ["Metric", "Value"]
print(tabulate(table_rows, headers=headers, tablefmt="grid", floatfmt=".2f"))

+---------------------------------+---------+
| Metric                          |   Value |
+=================================+=========+
| Average Accuracy                |    0.81 |
+---------------------------------+---------+
| Weighted Precision              |    0.83 |
+---------------------------------+---------+
| Weighted Recall                 |    0.81 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.25 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.05 |
+---------------------------------+---------+


## **KNN**

In [27]:
# Build Pipeline to process data separately during each fold
pipeline = Pipeline([
    ('imputation', OutlierReplacement()),
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Hyperparameter tuning, choose model with highest accuraccy acorss all values of k
results = GridSearchCV(estimator=pipeline, param_grid={"classifier__n_neighbors": [1, 3, 5]},
                       cv=cv, scoring=['accuracy', 'precision_weighted', 'recall_weighted'],
                       refit='accuracy',
                       return_train_score=False)

# Fit model
results.fit(sales, sales["category"])

# Extract scores for best value of k
knn_accuracy  = results.cv_results_['mean_test_accuracy'][results.best_index_]
knn_precision = results.cv_results_['mean_test_precision_weighted'][results.best_index_]
knn_recall    = results.cv_results_['mean_test_recall_weighted'][results.best_index_]
knn_train_time = results.cv_results_['mean_fit_time'][results.best_index_]
knn_test_time  = results.cv_results_['mean_score_time'][results.best_index_]

# Output scores
table_rows = [
    ["Average Accuracy", knn_accuracy],
    ["Weighted Precision", knn_precision],
    ["Weighted Recall", knn_recall],
    ["Average Training Time (Seconds)", knn_train_time],
    ["Average Training Time (Seconds)", knn_test_time]
]

headers = ["Metric", "Value"]
print(tabulate(table_rows, headers=headers, tablefmt="grid", floatfmt=".2f"))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+---------------------------------+---------+
| Metric                          |   Value |
+=================================+=========+
| Average Accuracy                |    0.81 |
+---------------------------------+---------+
| Weighted Precision              |    0.81 |
+---------------------------------+---------+
| Weighted Recall                 |    0.81 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.27 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.04 |
+---------------------------------+---------+


In [28]:
# Best value of k
print(results.best_params_['classifier__n_neighbors'])

3


## **Decision Tree**

In [29]:
# Build Pipeline to process data separately during each fold
pipeline = Pipeline([
    ('imputation', OutlierReplacement()),
    ('classifier', DecisionTreeClassifier())
])

# Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Hyperparameter tuning, choose model with highest accuraccy acorss all values of K
results = GridSearchCV(estimator=pipeline, param_grid={"classifier__max_depth": [5, 7, 10],
                                                       "classifier__min_samples_split": [2, 5, 7, 10]},
                       cv=cv, scoring=['accuracy', 'precision_weighted', 'recall_weighted'],
                       refit='accuracy',
                       return_train_score=False)

# Fit model
results.fit(sales, sales["category"])

# Extract scores for best value of K
tree_accuracy  = results.cv_results_['mean_test_accuracy'][results.best_index_]
tree_precision = results.cv_results_['mean_test_precision_weighted'][results.best_index_]
tree_recall    = results.cv_results_['mean_test_recall_weighted'][results.best_index_]
tree_train_time = results.cv_results_['mean_fit_time'][results.best_index_]
tree_test_time  = results.cv_results_['mean_score_time'][results.best_index_]

# Output scores
table_rows = [
    ["Average Accuracy", tree_accuracy],
    ["Weighted Precision", tree_precision],
    ["Weighted Recall", tree_recall],
    ["Average Training Time (Seconds)", tree_train_time],
    ["Average Training Time (Seconds)", tree_test_time]
]

headers = ["Metric", "Value"]
print(tabulate(table_rows, headers=headers, tablefmt="grid", floatfmt=".2f"))

+---------------------------------+---------+
| Metric                          |   Value |
+=================================+=========+
| Average Accuracy                |    0.90 |
+---------------------------------+---------+
| Weighted Precision              |    0.91 |
+---------------------------------+---------+
| Weighted Recall                 |    0.90 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.26 |
+---------------------------------+---------+
| Average Training Time (Seconds) |    0.04 |
+---------------------------------+---------+


In [30]:
# Best value of hyperparamters
print(results.best_params_['classifier__max_depth'])
print(results.best_params_['classifier__min_samples_split'])

5
2


## **Comparison of Classification Models**

In [31]:
# Side by side comparison of metrics for each model
table_rows = [
    ["Multinomial Logistic Regression", reg_accuracy, reg_precision, reg_recall, reg_train_time, reg_test_time],
    ["KNN", knn_accuracy, knn_precision, knn_recall, knn_train_time, knn_test_time],
    ["Decision Tree", tree_accuracy, tree_precision, tree_recall, tree_train_time, tree_test_time]
]
headers = ["Metric", "Average Accuracy", "Average Weighted Precision", "Average Weighted Recall", "Average Train Time (Seconds)", "Average Test Time (Seconds)"]
print(tabulate(table_rows, headers=headers, tablefmt="grid", floatfmt=".2f"))

+---------------------------------+--------------------+------------------------------+---------------------------+--------------------------------+-------------------------------+
| Metric                          |   Average Accuracy |   Average Weighted Precision |   Average Weighted Recall |   Average Train Time (Seconds) |   Average Test Time (Seconds) |
+=================================+====================+==============================+===========================+================================+===============================+
| Multinomial Logistic Regression |               0.81 |                         0.83 |                      0.81 |                           0.25 |                          0.05 |
+---------------------------------+--------------------+------------------------------+---------------------------+--------------------------------+-------------------------------+
| KNN                             |               0.81 |                         0.81 |        

## **Information Retrieval Pipeline**

In [32]:
# Filter data to include product descriptions and category
products = sales[["category", "about_product"]].copy()
products['id'] = [f'Doc{i+1:02d}' for i in range(len(products))]

In [33]:
# Apply text pre-processing

# Initialize resources
stop_en = set(stopwords.words('english'))
lemm = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_processing(text: str) -> str:
    text = str(text).lower()                                 # Lowercasing
    text = re.sub(r'\d+', ' num ', text)                     # Replace digits with the string 'num'
    tokens = word_tokenize(text)                             # Tokenization
    tokens = [t for t in tokens if t.isalpha()]              # Remove non-alphabetic tokens
    tokens = [t for t in tokens if t not in stop_en]         # Stopword removal
    tokens = [lemm.lemmatize(t) for t in tokens]             # Lemmatization
    tokens = [stemmer.stem(t) for t in tokens]               # Stemming
    return ' '.join(tokens)

# Pre-process each of the product descriptions
products['clean_about_product'] = products['about_product'].astype(str).apply(text_processing)

In [34]:
# Combined unigram and bigram TF-IDF
tfidf_unibi = TfidfVectorizer(ngram_range=(1,2), min_df=5, norm='l2')
X_unibi_tfidf = tfidf_unibi.fit_transform(products['clean_about_product'])

# Dimensions of matrix (rows = documents, columns = terms)
X_unibi_tfidf.shape

(449, 2504)

In [35]:
# Model evaluation

# Compute cosine similarity between all documents
similarities = cosine_similarity(X_unibi_tfidf, X_unibi_tfidf)

# Extract the top 20 for every product ranked similarities from largest to smallest
ranked_indices = np.argsort(-similarities, axis=1)[:, :20]

# Compute accuracy for each category
category_accuracies = {}

# Compare the categories for each of the related products retrived by the model
for product_index, product_row in enumerate(ranked_indices):
  matches = 0
  for similar_product_index in product_row:
      # Make we are not comparing the product to itself
      if product_index != similar_product_index:
        if products['category'].iloc[similar_product_index] == products['category'].iloc[product_index]:
          matches += 1

  # Add accuracy for each category
  if products['category'].iloc[product_index] in category_accuracies:
    category_accuracies[products['category'].iloc[product_index]].append(matches / 19)
  else:
    category_accuracies[products['category'].iloc[product_index]] = []
    category_accuracies[products['category'].iloc[product_index]].append(matches / 19)

average_category_accuracies = {}
# Compute mean retrieval accuracies by category
for category in category_accuracies:
  average_category_accuracies[category] = sum(category_accuracies[category]) / len(category_accuracies[category])



# Output accuracies
table_data = list(average_category_accuracies.items())
print(tabulate(table_data, headers=['Category', 'Average Accuracy'], tablefmt='grid', floatfmt=".2f"))

+-----------------------------------------------------------------------------------+--------------------+
| Category                                                                          |   Average Accuracy |
+===================================================================================+====================+
| Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables |               0.93 |
+-----------------------------------------------------------------------------------+--------------------+
| Electronics|HomeTheater,TV&Video|Televisions|SmartTelevisions                     |               1.00 |
+-----------------------------------------------------------------------------------+--------------------+
| Electronics|HomeTheater,TV&Video|Accessories|RemoteControls                       |               0.85 |
+-----------------------------------------------------------------------------------+--------------------+
| Electronics|WearableTechnology|Smar

In [36]:
!pip install nbconvert

In [37]:
!jupyter nbconvert --to html "/content/drive/MyDrive/CIND 820/Milestone 4/Final_Model_Development_Evaluation.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/CIND 820/Milestone 4/Final_Model_Development_Evaluation.ipynb to html
[NbConvertApp] Writing 351633 bytes to /content/drive/MyDrive/CIND 820/Milestone 4/Final_Model_Development_Evaluation.html
